# Machine Translation

In today's competition, your task is to implement the method described in the paper "Neural machine translation by jointly learning to align and translate". You will receive points for implementing and training the model, computing appropriate metrics, and conducting additional experiments and visualizations. During your work, you should ensure code clarity and readability, as well as the aesthetics of presented plots. Your implementation will be evaluated for compliance with the publication.

## Competition Rules

You must adhere to the following rules:

- You may not use the Internet. Exceptions are the OpenAI API, PyTorch documentation, and the Olympiad Google Classroom.
- You may not use Copilot or any other models that help write code, except for models from the GPT3.5 family.
- You may not use your own notes: both handwritten and files on the computer (including in particular code downloaded to the computer).
- You may not connect to computing resources other than Google Colab with T4 GPU.

## Task and scoring

You will train models on a dataset containing pairs of sentences in English and German (*parallel corpus*). Remember good coding practices and correct formatting, as this will affect your grade. Based on the attached paper, perform the following subtasks.

**Subtask 1: Model implementation (7 pts)**

In this section, we ask you to implement the method from the paper very precisely.

**Subtask 2: Model training (3 pts)**

Train the model on the provided dataset (dataloaders are already implemented in the starter code). During training, monitor both training and validation loss. Then create plots of these losses as a function of iteration. Evaluate the trained model on the test subset of the provided dataset using appropriate metrics. You can use the metrics used in the paper. Present examples of both correct and incorrect translations.

**Subtask 3: Attention visualization (1 pt)**

Present visualizations of learned attention maps on example interesting sentences. You can follow the plots from the paper.

**Subtask 4: Additional experiments (2 pts)**

If you have ideas for additional interesting experiments, you can get extra points for them. You may consider model modifications, ablations, and others.

## Notes
* You may change function and class signatures, as well as the code structure proposed by us below. However, remember good practices.

## Starter code

In [ ]:
! pip install datasets
! pip install evaluate
! pip install torchtext

# Download the embedding models
! python -m spacy download en_core_web_sm 
! python -m spacy download de_core_news_sm

In [ ]:
import random

import datasets
import evaluate
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import spacy
import torch
import torch.nn as nn

import torchtext; torchtext.disable_torchtext_deprecation_warning()
import torchtext.vocab; torchtext.disable_torchtext_deprecation_warning()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
######################### DO NOT CHANGE THIS CELL ##########################
seed = 1234

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True

## Datasets

We have prepared dataloaders and tokenizers for you.

In [ ]:
######################### DO NOT CHANGE THIS CELL ##########################
dataset = datasets.load_dataset("bentrevett/multi30k")

en_nlp = spacy.load("en_core_web_sm")
de_nlp = spacy.load("de_core_news_sm")

sos_token = "<sos>"
eos_token = "<eos>"


def tokenize_example(example, max_length=1000):
    en_tokens = [token.text.lower() for token in en_nlp.tokenizer(example["en"])][:max_length]
    de_tokens = [token.text.lower() for token in de_nlp.tokenizer(example["de"])][:max_length]

    en_tokens = [sos_token] + en_tokens + [eos_token]
    de_tokens = [sos_token] + de_tokens + [eos_token]
    return {"en_tokens": en_tokens, "de_tokens": de_tokens}


train_data = dataset["train"].map(tokenize_example)
valid_data = dataset["validation"].map(tokenize_example)
test_data = dataset["test"].map(tokenize_example)


In [ ]:
######################### DO NOT CHANGE THIS CELL ##########################
min_freq = 2
unk_token = "<unk>"
pad_token = "<pad>"

special_tokens = [unk_token, pad_token, sos_token, eos_token]

en_vocab = torchtext.vocab.build_vocab_from_iterator(
    train_data["en_tokens"],
    min_freq=min_freq,
    specials=special_tokens,
)

de_vocab = torchtext.vocab.build_vocab_from_iterator(
    train_data["de_tokens"],
    min_freq=min_freq,
    specials=special_tokens,
)

assert en_vocab[unk_token] == de_vocab[unk_token]
assert en_vocab[pad_token] == de_vocab[pad_token]

unk_index = en_vocab[unk_token]
pad_index = en_vocab[pad_token]

en_vocab.set_default_index(unk_index)
de_vocab.set_default_index(unk_index)

In [ ]:
######################### DO NOT CHANGE THIS CELL ##########################
def numericalize_example(example):
    en_ids = en_vocab.lookup_indices(example["en_tokens"])
    de_ids = de_vocab.lookup_indices(example["de_tokens"])
    return {"en_ids": en_ids, "de_ids": de_ids}

train_data = train_data.map(numericalize_example)
valid_data = valid_data.map(numericalize_example)
test_data = test_data.map(numericalize_example)

In [ ]:
######################### DO NOT CHANGE THIS CELL ##########################
format_columns = ["en_ids", "de_ids"]

train_data = train_data.with_format(
    type="torch", columns=format_columns, output_all_columns=True
)

valid_data = valid_data.with_format(
    type="torch",
    columns=format_columns,
    output_all_columns=True,
)

test_data = test_data.with_format(
    type="torch",
    columns=format_columns,
    output_all_columns=True,
)

In [ ]:
######################### DO NOT CHANGE THIS CELL ##########################
BATCH_SIZE = 128

def get_data_loader(dataset, batch_size, pad_index, shuffle=False):
    def collate_fn(batch):
        batch_en_ids = [example["en_ids"] for example in batch]
        batch_de_ids = [example["de_ids"] for example in batch]
        batch_en_ids = nn.utils.rnn.pad_sequence(batch_en_ids, padding_value=pad_index)
        batch_de_ids = nn.utils.rnn.pad_sequence(batch_de_ids, padding_value=pad_index)
        batch = {
            "en_ids": batch_en_ids,
            "de_ids": batch_de_ids,
        }
        return batch

    data_loader = torch.utils.data.DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        collate_fn=collate_fn,
        shuffle=shuffle,
    )
    return data_loader

train_data_loader = get_data_loader(train_data, BATCH_SIZE, pad_index, shuffle=True)
valid_data_loader = get_data_loader(valid_data, BATCH_SIZE, pad_index)
test_data_loader = get_data_loader(test_data, BATCH_SIZE, pad_index)

## Subtask 1: Model implementation

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, embedding_dim, encoder_hidden_dim, decoder_hidden_dim, dropout):
        super().__init__()
        # TODO

    def forward(self, src):
        # TODO
        return outputs, hidden

In [ ]:
class Attention(nn.Module):
    def __init__(self, encoder_hidden_dim, decoder_hidden_dim):
        super().__init__()
        # TODO

    def forward(self, hidden, encoder_outputs):
        # TODO
        return result

In [ ]:
class Decoder(nn.Module):
    def __init__(
        self,
        output_dim,
        embedding_dim,
        encoder_hidden_dim,
        decoder_hidden_dim,
        dropout,
        attention,
    ):
        super().__init__()
        # TODO

    def forward(self, input, hidden, encoder_outputs):
        # TODO
        return prediction, hidden, attention

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        # TODO

    def forward(self, src, trg):
        # TODO
        return outputs

In [ ]:
input_dim = len(de_vocab)
output_dim = len(en_vocab)
encoder_embedding_dim = 256
decoder_embedding_dim = 256
encoder_hidden_dim = 512
decoder_hidden_dim = 512
encoder_dropout = 0.5
decoder_dropout = 0.5

attention = ...  # TODO
encoder = ...  # TODO
decoder = ...  # TODO
model = ...  # TODO

## Subtask 2: Model training

In [4]:
# TODO: Write the training loop. Remember to collect training and validation statistics.

# TODO: Train the model on the provided dataset

In [ ]:
# TODO: Create loss function plots

In [ ]:
# TODO: Evaluate the model on the test set

In [ ]:
def translate_sentence(sentence, model):
    # TODO
    return en_tokens, de_tokens, attention


# TODO: Examples of translations

## Subtask 3: Attention visualization

In [ ]:
def plot_attention(sentence, translation, attention):
    # TODO


# TODO: Attention visualization

## Subtask 4: Additional experiments


In [ ]:
# TODO